In [1]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee




# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


🚀 MCP Demo Started (Type 'exit' to stop)

Enter your query: i am facing high network ping

🧠 Agent received input: i am facing high network ping
➡️ Decision: No tool needed (AI response)
🤖 AI Response: Please describe your issue clearly.
Enter your query:  i am having high network ping due to which i can't work properly

🧠 Agent received input:  i am having high network ping due to which i can't work properly
➡️ Decision: No tool needed (AI response)
🤖 AI Response: Please describe your issue clearly.
Enter your query: vpn issue

🧠 Agent received input: vpn issue
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: medium
📦 MCP Payload: {'issue': 'vpn issue', 'priority': 'medium', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: vpn issue
        Category: network
        Priority: medium
        
Enter your query: network issue

🧠 Agent received input: network issue
➡️ Decision:

KeyboardInterrupt: Interrupted by user

In [5]:
!pip install groq
import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("GROQ_API_KEY")
print(api_key)

client = Groq(api_key=api_key)

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: LLM ANALYSIS (REPLACES RULES)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - should_create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an IT helpdesk assistant.

Analyze the user issue and respond in JSON format:

{{
  "create_ticket": true/false,
  "category": "network/hardware/software/general",
  "priority": "high/medium/low"
}}

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # fast + powerful
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 3: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        return "🤖 AI Response: No ticket required. Try basic troubleshooting."


# ============================================
# STEP 4: RUN LOOP
# ============================================

print("🚀 LLM MCP Helpdesk Started (type 'exit')\n")

while True:

    user_input = input("Enter issue: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(user_input)
    print(response)

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
🚀 LLM MCP Helpdesk Started (type 'exit')


🧠 Agent received: high network issue
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'high network issue', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: high network issue
Category: general
Priority: medium


🧠 Agent received: facing glitch in laptop please review it immediately
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'facing glitch in laptop please review it immediately', 'priority': 'medium', 'category': 'general'}

✅ Ticket Created Successfully!

Ticket ID: INC1001
Issue: facing glitch in laptop please review it immediately
Category: general
Priority: medium



KeyboardInterrupt: Interrupted by user

In [6]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”
# 🧩 Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
# 👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

# 🤖 What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  |
# |  |  |
# |  |  |
# |  |  |
# |  |  |



# This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?


🚀 HR Onboarding MCP Demo Started (type 'exit' to stop)

Enter your query: email is not working

🧠 Agent received input: email is not working
📊 Extracted → Category: it setup, Priority: medium
➡️ Decision: Escalation required (Tool call)
📦 MCP Payload: {'issue': 'email is not working', 'priority': 'medium', 'category': 'it setup'}
⚙️ HR Ticket Created

        ✅ HR Ticket Created Successfully!

        🆔 Ticket ID: HR2000
        📝 Issue: email is not working
        📂 Category: it setup
        ⚡ Priority: medium

        👉 HR team will contact you shortly.
        


KeyboardInterrupt: Interrupted by user

In [8]:
!pip install groq

import os
from groq import Groq
from google.colab import userdata

# Load API key securely
api_key = userdata.get("GROQ_API_KEY")
print(api_key)

client = Groq(api_key=api_key)


# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL (HR TICKET SYSTEM)
# ============================================

def create_hr_ticket(issue, priority, category):
    ticket_id = f"HR{2000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: KNOWLEDGE BASE (FAQ)
# ============================================

def get_faq_response(category):

    faqs = {
        "payroll": "💰 Payroll Help: Login via payroll portal. Use employee ID. Reset password if needed.",
        "policy": "📜 Leave Policy: 20 annual + 10 sick leaves. Apply via HR portal.",
        "it setup": "💻 IT Setup: Use company email credentials. Contact IT if login fails.",
        "training": "📚 Training: Check onboarding dashboard → 'My Training'.",
        "general": "🤖 Please provide more details so I can help."
    }

    return faqs.get(category, faqs["general"])


# ============================================
# STEP 3: LLM ANALYSIS (CORE MCP INTELLIGENCE)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - create_ticket
    - category
    - priority
    """

    prompt = f"""
You are an HR onboarding assistant.

Analyze the employee query and respond STRICTLY in JSON:

{{
  "create_ticket": true/false,
  "category": "payroll/policy/it setup/training/general",
  "priority": "high/medium"
}}

Rules:
- Create ticket ONLY if issue is blocking (login failure, system error, not working)
- Otherwise give guidance (no ticket)

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 4: MCP AGENT
# ============================================

def mcp_hr_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_hr_ticket(**payload)

        return f"""
✅ HR Ticket Created Successfully!

🆔 Ticket ID: {result['ticket_id']}
📝 Issue: {result['issue']}
📂 Category: {result['category']}
⚡ Priority: {result['priority']}

👉 HR team will contact you shortly.
"""

    else:

        faq = get_faq_response(decision["category"])

        return f"""
🤖 Instant Help:

{faq}

👉 If this doesn't resolve your issue, describe your problem again with details.
"""


# ============================================
# STEP 5: RUN LOOP
# ============================================

print("🚀 LLM MCP HR Onboarding Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_hr_agent(user_input)
    print(response)

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
🚀 LLM MCP HR Onboarding Assistant Started (type 'exit')


🧠 Agent received: my email is not sen\ding having configuration issue
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'my email is not sen\\ding having configuration issue', 'priority': 'medium', 'category': 'general'}

✅ HR Ticket Created Successfully!

🆔 Ticket ID: HR2000
📝 Issue: my email is not sen\ding having configuration issue
📂 Category: general
⚡ Priority: medium

👉 HR team will contact you shortly.



KeyboardInterrupt: Interrupted by user

In [9]:
# MCP SCENARIO: “Smart Banking Support Assistant”
# 🧩 Scenario Background
# You are working in a company called FinTrust Bank.
# Customers often face issues such as:
# - Credit card not working
# - Trouble with online banking login
# - Queries about loan status
# - Transaction disputes
# 👉 Instead of calling customer care, customers use an AI Banking Support Bot.

# 🤖 What this Bot Should Do
# When a customer types a problem:
# - Understand the issue (e.g., “My card was declined”)
# - Decide if escalation to a human agent is needed
# - Identify:
# - Category (Card Services / Online Banking / Loans / Transactions)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, troubleshooting steps, policy info)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# |  |  |
# |  |  |
# |  |  |
# |  |  |
# |  |  |



# This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
# Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

# 🧩 Company: FinTrust Bank


# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL (BANK SUPPORT SYSTEM)
# ============================================

def create_ticket(issue, priority, category):
    """
    Simulates Banking Support Ticket System
    """

    ticket_id = f"BNK{3000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: KNOWLEDGE BASE (FAQ RESPONSES)
# ============================================

def get_faq_response(category):

    faqs = {
        "card services": "💳 Card Help: Ensure your card is active and has sufficient balance. Try re-entering PIN or enable online usage.",
        "online banking": "🌐 Login Help: Reset your password using 'Forgot Password'. Ensure correct username.",
        "loans": "🏦 Loan Status: Check loan status in 'My Loans' section of mobile app or net banking.",
        "transactions": "💰 Transaction Help: Pending transactions may take 24 hours. Check statement or contact support.",
        "general": "🤖 Please provide more details so I can assist you."
    }

    return faqs.get(category, faqs["general"])


# ============================================
# STEP 3: AGENT REASONING (RULE-BASED)
# ============================================

def analyze_input(user_input):

    text = user_input.lower()

    # 🔹 Category Detection
    if "card" in text or "credit card" in text or "debit card" in text:
        category = "card services"
    elif "login" in text or "password" in text or "net banking" in text:
        category = "online banking"
    elif "loan" in text or "emi" in text:
        category = "loans"
    elif "transaction" in text or "payment" in text or "failed" in text:
        category = "transactions"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "fraud" in text or "blocked" in text:
        priority = "high"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 4: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):

    escalation_keywords = [
        "not working",
        "failed",
        "declined",
        "blocked",
        "fraud",
        "issue",
        "problem",
        "error"
    ]

    return any(word in user_input.lower() for word in escalation_keywords)


# ============================================
# STEP 5: MCP ORCHESTRATOR
# ============================================

def mcp_banking_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # STEP 5.1: Analyze input
    category, priority = analyze_input(user_input)
    print(f"📊 Extracted → Category: {category}, Priority: {priority}")

    # STEP 5.2: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Escalation required (Tool call)")

        # STEP 5.3: MCP Payload
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 5.4: Tool execution
        result = create_ticket(**payload)
        print("⚙️ Banking Ticket Created")

        # STEP 5.5: Response
        return f"""
✅ Support Ticket Created Successfully!

🆔 Ticket ID: {result['ticket_id']}
📝 Issue: {result['issue']}
📂 Category: {result['category']}
⚡ Priority: {result['priority']}

👉 Our banking support team will contact you shortly.
"""

    else:
        print("➡️ Decision: No escalation (AI response)")

        # STEP 5.6: Instant help
        faq = get_faq_response(category)

        return f"""
🤖 Instant Help:

{faq}

👉 If your issue persists, mention 'problem' or 'not working' to create a ticket.
"""


# ============================================
# STEP 6: RUN INTERACTIVE LOOP
# ============================================

print("🚀 Banking MCP Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting Banking MCP...")
        break

    response = mcp_banking_agent(user_input)
    print(response)

🚀 Banking MCP Assistant Started (type 'exit')

Enter your query: my card is blocked

🧠 Agent received: my card is blocked
📊 Extracted → Category: card services, Priority: high
➡️ Decision: Escalation required (Tool call)
📦 MCP Payload: {'issue': 'my card is blocked', 'priority': 'high', 'category': 'card services'}
⚙️ Banking Ticket Created

✅ Support Ticket Created Successfully!

🆔 Ticket ID: BNK3000
📝 Issue: my card is blocked
📂 Category: card services
⚡ Priority: high

👉 Our banking support team will contact you shortly.



KeyboardInterrupt: Interrupted by user

In [10]:
!pip install groq

import os
from groq import Groq
from google.colab import userdata

# Load API key
api_key = userdata.get("GROQ_API_KEY")
print(api_key)

client = Groq(api_key=api_key)


# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []


# ============================================
# STEP 1: TOOL (BANKING TICKET SYSTEM)
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{3000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket


# ============================================
# STEP 2: KNOWLEDGE BASE (FAQ)
# ============================================

def get_faq_response(category):

    faqs = {
        "card services": "💳 Check if your card is active, has balance, and online transactions are enabled.",
        "online banking": "🌐 Reset your password using 'Forgot Password' and ensure correct login credentials.",
        "loans": "🏦 Visit 'My Loans' section in app to check status or EMI details.",
        "transactions": "💰 Transactions may take up to 24 hours. Check statement or retry.",
        "general": "🤖 Please provide more details so I can assist you."
    }

    return faqs.get(category, faqs["general"])


# ============================================
# STEP 3: LLM ANALYSIS (CORE MCP INTELLIGENCE)
# ============================================

def analyze_with_llm(user_input):
    """
    LLM decides:
    - create_ticket
    - category
    - priority
    """

    prompt = f"""
You are a banking support assistant for FinTrust Bank.

Analyze the customer query and respond STRICTLY in JSON:

{{
  "create_ticket": true/false,
  "category": "card services/online banking/loans/transactions/general",
  "priority": "high/medium"
}}

Rules:
- HIGH priority if fraud, blocked card, login failure, payment failure
- Create ticket ONLY if issue is serious or unresolved
- Otherwise provide guidance

User Input: "{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    output = response.choices[0].message.content

    try:
        import json
        parsed = json.loads(output)
    except:
        parsed = {
            "create_ticket": True,
            "category": "general",
            "priority": "medium"
        }

    return parsed


# ============================================
# STEP 4: MCP AGENT
# ============================================

def mcp_banking_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # LLM Decision
    decision = analyze_with_llm(user_input)

    print("🤖 LLM Decision:", decision)

    if decision["create_ticket"]:

        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"]
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        return f"""
✅ Support Ticket Created Successfully!

🆔 Ticket ID: {result['ticket_id']}
📝 Issue: {result['issue']}
📂 Category: {result['category']}
⚡ Priority: {result['priority']}

👉 Our banking team will contact you shortly.
"""

    else:

        faq = get_faq_response(decision["category"])

        return f"""
🤖 Instant Help:

{faq}

👉 If the issue continues, describe the problem again to raise a ticket.
"""


# ============================================
# STEP 5: RUN LOOP
# ============================================

print("🚀 LLM MCP Banking Assistant Started (type 'exit')\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_banking_agent(user_input)
    print(response)

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
🚀 LLM MCP Banking Assistant Started (type 'exit')

Enter your query: my card has been blocked

🧠 Agent received: my card has been blocked
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'my card has been blocked', 'priority': 'medium', 'category': 'general'}

✅ Support Ticket Created Successfully!

🆔 Ticket ID: BNK3000
📝 Issue: my card has been blocked
📂 Category: general
⚡ Priority: medium

👉 Our banking team will contact you shortly.

Enter your query: my accoung has hacked

🧠 Agent received: my accoung has hacked
🤖 LLM Decision: {'create_ticket': True, 'category': 'general', 'priority': 'medium'}
📦 MCP Payload: {'issue': 'my accoung has hacked', 'priority': 'medium', 'category': 'general'}

✅ Support Ticket Created Successfully!

🆔 Ticket ID: BNK3001
📝 Issue: my accoung has hacked
📂 Category: general
⚡ Priority: medium

👉 Our banking team will contact you shortly.



KeyboardInterrupt: Interrupted by user

In [ ]:
# ============================================
# MCP WEATHER TOOL SERVER
# ============================================

import requests

# ============================================
# STEP 0: CONFIG
# ============================================

API_KEY = "YOUR_OPENWEATHER_API_KEY"   # 🔑 Replace this
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"


# ============================================
# STEP 1: TOOL (WEATHER API)
# ============================================

def get_weather(city):
    """
    MCP TOOL → Fetch weather data from API
    """

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        return {"error": "City not found"}

    data = response.json()

    return {
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "condition": data["weather"][0]["description"],
        "humidity": data["main"]["humidity"]
    }


# ============================================
# STEP 2: MCP TOOL REGISTRY
# ============================================

TOOLS = {
    "get_weather": {
        "function": get_weather,
        "description": "Get current weather of a city",
        "params": ["city"]
    }
}


# ============================================
# STEP 3: MCP DECISION ENGINE
# ============================================

def should_call_tool(user_input):
    """
    Decide if weather tool is needed
    """

    keywords = ["weather", "temperature", "rain", "forecast"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: PARAMETER EXTRACTION
# ============================================

def extract_city(user_input):
    """
    Simple extraction of city name
    """

    words = user_input.split()

    # naive approach: last word = city
    return words[-1]


# ============================================
# STEP 5: MCP SERVER (ORCHESTRATOR)
# ============================================

def mcp_weather_agent(user_input):

    print("\n🧠 Agent received:", user_input)

    # STEP 5.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Call Weather Tool")

        # STEP 5.2: Extract params
        city = extract_city(user_input)

        payload = {
            "city": city
        }

        print("📦 MCP Payload:", payload)

        # STEP 5.3: Tool call
        result = TOOLS["get_weather"]["function"](**payload)

        print("⚙️ Tool executed")

        if "error" in result:
            return "❌ Could not fetch weather. Check city name."

        # STEP 5.4: Response
        return f"""
🌤️ Weather in {result['city']}

🌡️ Temperature: {result['temperature']}°C
☁️ Condition: {result['condition']}
💧 Humidity: {result['humidity']}%
"""

    else:
        return "🤖 Ask me about weather like: 'Weather in Delhi'"


# ============================================
# STEP 6: RUN SERVER LOOP
# ============================================

print("🚀 MCP Weather Server Started (type 'exit')\n")

while True:

    user_input = input("Enter query: ")

    if user_input.lower() == "exit":
        print("👋 Server stopped")
        break

    response = mcp_weather_agent(user_input)
    print(response)